# NetShield — UNSW-NB15 Dataset Exploration

## CS-245 Machine Learning

### Project
**NetShield: Investigating Dataset-Dependent Behavior in Machine Learning-Based Network Intrusion Detection**

### Purpose
This notebook performs the **exploratory data analysis (EDA)** of the UNSW-NB15 dataset before preprocessing and model training.

The exploration is designed to support the project objectives of comparing datasets, understanding feature and traffic-distribution differences, investigating class imbalance, and later building within-dataset and cross-dataset IDS experiments.

### EDA objectives
1. Verify the dataset files and predefined train/test split.
2. Understand dataset dimensions, columns, data types, and feature meanings.
3. Check data quality: missing values, duplicates, infinite values, suspicious placeholders, and constant features.
4. Analyze binary labels and attack categories, including class imbalance.
5. Examine numerical feature ranges, skewness, zero prevalence, and potential outliers.
6. Examine feature correlations and association with the binary target.
7. Compare important feature distributions and target distributions between training and testing data.
8. Check exact train/test record overlap as an additional leakage-risk diagnostic.
9. Save reusable EDA tables for later preprocessing, feature mapping, and experimentation.

> **Important:** This notebook is intentionally exploratory. It does **not** remove rows, impute values, scale features, encode categories, balance classes, or train ML models. Those decisions belong to the preprocessing/modeling stages and must be handled using training data only where applicable.

## 1. Imports and display settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from IPython.display import display

# Optional statistical test used later for train/test distribution comparison.
try:
    from scipy.stats import ks_2samp
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")
print("SciPy available:", SCIPY_AVAILABLE)

## 2. Project paths

Expected project structure:

```text
NetShield/
├── data/
│   └── raw/
│       └── UNSW-NB15/
└── notebooks/
    └── 01_UNSW_exploration.ipynb
```

The project root is located automatically so the notebook is less dependent on the current working directory.

In [ ]:
def find_project_root():
    """Find the nearest project directory containing data/raw/UNSW-NB15."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "UNSW-NB15").exists():
            return candidate
    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    # Standard fallback if the notebook is inside NetShield/notebooks.
    PROJECT_ROOT = (Path.cwd() / "..").resolve()

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "UNSW_NB15_EDA"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)
print("\nUNSW-NB15 directory:")
print(DATA_DIR)
print("\nUNSW EDA output directory:")
print(RESULTS_DIR)
print("\nFigures directory:")
print(FIGURES_DIR)

In [ ]:
# ============================================================
# EDA Output / Figure Saving Helper
# ============================================================

import re

def save_current_figure(filename):
    """Save the current matplotlib figure in the UNSW EDA figures folder."""
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", str(filename)).strip("_")
    if not safe_name.lower().endswith(".png"):
        safe_name += ".png"
    filepath = FIGURES_DIR / safe_name
    plt.savefig(filepath, dpi=180, bbox_inches="tight")
    print(f"Figure saved: {filepath}")

print("UNSW EDA outputs will be stored in:", RESULTS_DIR)
print("UNSW EDA figures will be stored in:", FIGURES_DIR)


## 3. Verify the dataset directory

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        "UNSW-NB15 directory was not found. Expected location: " + str(DATA_DIR.resolve())
    )

print("✅ UNSW-NB15 directory found.")

## 4. Inventory the downloaded CSV files

Windows may display CSV files as an Excel/XLS-associated type. The code below checks the actual `.csv` extension directly.

The project download contains the predefined training/testing files as well as the original/auxiliary UNSW-NB15 CSV resources. We use the **predefined training/testing split for the main EDA** because it is the split intended for reproducible train/test experimentation.

In [ ]:
csv_files = sorted(
    f for f in DATA_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() == ".csv"
)

print(f"Number of CSV files found: {len(csv_files)}\n")
for file in csv_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{size_mb:9.1f} MB   {file.relative_to(DATA_DIR)}")

## 5. Locate the predefined training and testing sets

In [ ]:
training_files = [f for f in csv_files if "training" in f.name.lower()]
testing_files = [f for f in csv_files if "testing" in f.name.lower()]

print("Training files:")
for f in training_files:
    print(" -", f)

print("\nTesting files:")
for f in testing_files:
    print(" -", f)

if not training_files or not testing_files:
    raise FileNotFoundError(
        "Could not locate both predefined training and testing CSV files. "
        "Check the file inventory above and adjust the filename filter if necessary."
    )

# The standard download contains one predefined training file and one predefined testing file.
train_path = training_files[0]
test_path = testing_files[0]

print("\nSelected training file:", train_path)
print("Selected testing file:", test_path)

## 6. Load the predefined train/test datasets

In [ ]:
train_df = pd.read_csv(train_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)

print("Training dataset loaded successfully.")
print("Testing dataset loaded successfully.")
print("\nTraining shape:", train_df.shape)
print("Testing shape: ", test_df.shape)

## 7. First look at the data

In [ ]:
display(train_df.head())

In [ ]:
display(test_df.head())

## 8. Dataset dimensions and basic structure

For this analysis, one row represents a network-flow record and the columns describe properties of that flow or its label/category.

In [ ]:
print("TRAINING DATA")
print("=" * 60)
print(f"Rows:    {train_df.shape[0]:,}")
print(f"Columns: {train_df.shape[1]:,}")

print("\nTESTING DATA")
print("=" * 60)
print(f"Rows:    {test_df.shape[0]:,}")
print(f"Columns: {test_df.shape[1]:,}")

print("\nMemory usage:")
print(f"Training: {train_df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")
print(f"Testing:  {test_df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")

## 9. Column inventory and train/test consistency

In [ ]:
print("UNSW-NB15 columns")
print("=" * 70)
for i, column in enumerate(train_df.columns, start=1):
    print(f"{i:2}. {column}")

train_columns = set(train_df.columns)
test_columns = set(test_df.columns)

print("\nColumns only in training:", sorted(train_columns - test_columns))
print("Columns only in testing: ", sorted(test_columns - train_columns))
print("Same column set:          ", train_columns == test_columns)

## 10. Data types and feature information table

In [ ]:
feature_info = pd.DataFrame({
    "Feature": train_df.columns,
    "Data_Type": train_df.dtypes.astype(str).values,
    "Missing_Count": train_df.isnull().sum().values,
    "Missing_Percentage": (train_df.isnull().mean().values * 100),
    "Unique_Values": train_df.nunique(dropna=False).values,
})

display(feature_info)

## 11. Feature roles

The target columns are kept separate from the traffic features. This distinction is important so that target information is not accidentally treated as an input feature during later modeling.

In [ ]:
TARGET_COLUMNS = [c for c in ["label", "attack_cat"] if c in train_df.columns]
ID_COLUMNS = [c for c in ["id"] if c in train_df.columns]

numeric_columns = train_df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = train_df.select_dtypes(exclude=np.number).columns.tolist()

numeric_feature_columns = [
    c for c in numeric_columns
    if c not in TARGET_COLUMNS + ID_COLUMNS
]

categorical_feature_columns = [
    c for c in categorical_columns
    if c not in TARGET_COLUMNS + ID_COLUMNS
]

all_feature_columns = numeric_feature_columns + categorical_feature_columns

print("Target columns:", TARGET_COLUMNS)
print("Identifier columns:", ID_COLUMNS)
print("\nNumerical traffic features:", len(numeric_feature_columns))
print(numeric_feature_columns)
print("\nCategorical traffic features:", len(categorical_feature_columns))
print(categorical_feature_columns)

## 12. Official feature-description file

The downloaded UNSW-NB15 package includes a feature-description CSV. We inspect it here so that later feature mapping to CICIDS2017 is based on **feature meaning**, not just similar column names.

If the local package uses a slightly different filename, the code searches for any CSV containing `feature` in its filename.

In [ ]:
feature_definition_files = [
    f for f in csv_files
    if "feature" in f.name.lower()
]

if feature_definition_files:
    feature_definition_path = feature_definition_files[0]
    print("Feature-definition file:", feature_definition_path)
    try:
        feature_definitions = pd.read_csv(feature_definition_path, low_memory=False)
        print("Shape:", feature_definitions.shape)
        display(feature_definitions.head(20))
        print("\nColumns:", feature_definitions.columns.tolist())
    except Exception as e:
        feature_definitions = None
        print("Could not parse the feature-definition file automatically:", e)
else:
    feature_definitions = None
    print("No feature-definition CSV was found in the downloaded package.")

### Feature-definition inspection

The table above is retained as a reference for the later semantic feature-alignment stage. No feature is dropped or renamed here.

## 13. Missing-value analysis

In [ ]:
missing_train = train_df.isnull().sum().sort_values(ascending=False)
missing_test = test_df.isnull().sum().sort_values(ascending=False)

print("Missing values — training:")
display(missing_train[missing_train > 0].to_frame("Missing_Count") if (missing_train > 0).any() else pd.DataFrame({"Result": ["None found"]}))

print("Missing values — testing:")
display(missing_test[missing_test > 0].to_frame("Missing_Count") if (missing_test > 0).any() else pd.DataFrame({"Result": ["None found"]}))

## 14. Duplicate-record analysis

Duplicates are **measured, not removed**. The second check ignores the identifier because an `id` can make otherwise identical flow records appear unique.

In [ ]:
duplicate_full_train = int(train_df.duplicated().sum())
duplicate_full_test = int(test_df.duplicated().sum())

cols_without_id = [c for c in train_df.columns if c.lower() != "id"]
duplicate_no_id_train = int(train_df.duplicated(subset=cols_without_id).sum())
duplicate_no_id_test = int(test_df.duplicated(subset=cols_without_id).sum())

print(f"Exact duplicate rows — training: {duplicate_full_train:,}")
print(f"Exact duplicate rows — testing:  {duplicate_full_test:,}")
print(f"Duplicate rows ignoring id — training: {duplicate_no_id_train:,}")
print(f"Duplicate rows ignoring id — testing:  {duplicate_no_id_test:,}")

## 15. Infinite and non-finite numeric values

In [ ]:
inf_train = np.isinf(train_df[numeric_columns]).sum().sort_values(ascending=False)
inf_test = np.isinf(test_df[numeric_columns]).sum().sort_values(ascending=False)

print("Infinite values — training:")
display(inf_train[inf_train > 0].to_frame("Infinite_Count") if (inf_train > 0).any() else pd.DataFrame({"Result": ["None found"]}))

print("Infinite values — testing:")
display(inf_test[inf_test > 0].to_frame("Infinite_Count") if (inf_test > 0).any() else pd.DataFrame({"Result": ["None found"]}))

## 16. Categorical-value inspection

Categorical variables are inspected without encoding them. This helps identify low-frequency categories and placeholders that will need deliberate treatment during preprocessing.

In [ ]:
for col in categorical_feature_columns:
    print("\n" + "=" * 75)
    print(f"FEATURE: {col} | Unique values: {train_df[col].nunique(dropna=False):,}")
    print("=" * 75)
    display(train_df[col].value_counts(dropna=False).head(20).to_frame("Count"))

## 17. Suspicious placeholders in categorical columns

Values such as `-` can be legitimate dataset conventions. They are therefore reported rather than automatically treated as missing.

In [ ]:
suspicious_tokens = {"nan", "none", "null", "inf", "-inf", "infinity", "-infinity", "", "-"}
suspicious_rows = []

for col in categorical_feature_columns:
    normalized = train_df[col].astype(str).str.strip().str.lower()
    for token in suspicious_tokens:
        count = int((normalized == token).sum())
        if count:
            suspicious_rows.append({
                "Feature": col,
                "Value": token,
                "Count": count,
                "Percentage": count / len(train_df) * 100,
            })

suspicious_summary = pd.DataFrame(suspicious_rows)
if suspicious_summary.empty:
    print("No suspicious string placeholders were found.")
else:
    display(suspicious_summary.sort_values("Count", ascending=False))

if "service" in train_df.columns:
    print("\nNote: '-' in UNSW-NB15's service field is commonly used as a 'no service identified' placeholder and should not automatically be treated as corrupted data.")

## 18. Descriptive statistics for numerical features

In [ ]:
display(train_df[numeric_feature_columns].describe().T)

In [ ]:
detailed_numeric_summary = train_df[numeric_feature_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(detailed_numeric_summary)

## 19. Zero-value analysis

Zeros are not automatically treated as missing values. In network-flow data, a zero can represent a genuine absence of activity for a particular measurement. We quantify zero prevalence so that later preprocessing decisions are evidence-based.

In [ ]:
zero_summary = pd.DataFrame({
    "Feature": numeric_feature_columns,
    "Zero_Count": [(train_df[c] == 0).sum() for c in numeric_feature_columns],
})
zero_summary["Zero_Percentage"] = zero_summary["Zero_Count"] / len(train_df) * 100
zero_summary = zero_summary.sort_values("Zero_Percentage", ascending=False).reset_index(drop=True)

display(zero_summary)

## 20. Quantitative skewness analysis

In [ ]:
skewness_summary = train_df[numeric_feature_columns].skew(numeric_only=True).sort_values(key=lambda s: s.abs(), ascending=False)
skewness_table = skewness_summary.to_frame("Skewness")
skewness_table["Absolute_Skewness"] = skewness_table["Skewness"].abs()

display(skewness_table)

In [ ]:
top_skewed = skewness_summary.abs().sort_values(ascending=False).head(15).index

plt.figure(figsize=(10, 6))
skewness_summary.loc[top_skewed].sort_values().plot(kind="barh")
plt.title("Most Skewed Numerical Features — UNSW-NB15")
plt.xlabel("Skewness")
plt.ylabel("Feature")
plt.tight_layout()
save_current_figure("01_most_skewed_numerical_features.png")
plt.show()

## 21. Binary target: normal vs attack

In [ ]:
if "label" not in train_df.columns:
    print("The expected 'label' column was not found.")
else:
    label_counts = train_df["label"].value_counts().sort_index()
    label_percentages = train_df["label"].value_counts(normalize=True).sort_index() * 100

    label_summary = pd.DataFrame({
        "Count": label_counts,
        "Percentage": label_percentages.round(2),
    })
    label_summary.index.name = "Label"
    display(label_summary)

    if len(label_counts) >= 2:
        imbalance_ratio = label_counts.max() / label_counts.min()
        print(f"Majority/minority class ratio: {imbalance_ratio:.2f}:1")

    plt.figure(figsize=(7, 5))
    label_counts.sort_index().plot(kind="bar")
    plt.title("UNSW-NB15 Training Data — Normal vs Attack")
    plt.xlabel("Label (0 = Normal, 1 = Attack)")
    plt.ylabel("Number of Records")
    plt.xticks(rotation=0)
    plt.tight_layout()
    save_current_figure("02_normal_vs_attack_distribution.png")
    plt.show()


## 22. Attack-category distribution

`attack_cat` describes the attack family. In the raw UNSW-NB15 representation, normal records may have a blank/NaN attack category. For EDA only, we create a separate display label called `attack_group` so that **Normal traffic is included in the same distribution as the attack categories**. The original dataframe is not modified.

In [ ]:
if "attack_cat" in train_df.columns:
    attack_group = train_df["attack_cat"].fillna("Normal").astype(str).str.strip()
    attack_group = attack_group.replace({"": "Normal", "nan": "Normal", "NaN": "Normal"})

    attack_group_counts = attack_group.value_counts()
    attack_group_summary = pd.DataFrame({
        "Count": attack_group_counts,
        "Percentage": (attack_group_counts / len(train_df) * 100).round(2),
    })

    display(attack_group_summary)

    plt.figure(figsize=(12, 6))
    attack_group_counts.sort_values().plot(kind="barh")
    plt.title("UNSW-NB15 Training Data — Normal and Attack Categories")
    plt.xlabel("Number of Records")
    plt.ylabel("Category")
    plt.tight_layout()
    save_current_figure("03_normal_and_attack_categories.png")
    plt.show()
else:
    attack_group = None
    attack_group_summary = pd.DataFrame()
    print("The expected 'attack_cat' column was not found.")


## 23. Attack-only category distribution

This view excludes normal traffic and focuses only on the relative frequencies of the malicious categories. It is useful for identifying minority attack types.

In [ ]:
if "attack_cat" in train_df.columns:
    attack_only = train_df["attack_cat"].dropna().astype(str).str.strip()
    attack_only = attack_only[attack_only != ""]

    attack_counts = attack_only.value_counts()
    attack_summary = pd.DataFrame({
        "Count": attack_counts,
        "Percentage_of_Attacks": (attack_counts / attack_counts.sum() * 100).round(2),
        "Percentage_of_All_Records": (attack_counts / len(train_df) * 100).round(2),
    })

    display(attack_summary)

    if len(attack_counts):
        print("Largest attack category:", attack_counts.index[0])
        print("Smallest attack category/categories:", list(attack_counts[attack_counts == attack_counts.min()].index))
else:
    attack_summary = pd.DataFrame()

## 24. Train vs test target distributions

In [ ]:
if "label" in train_df.columns and "label" in test_df.columns:
    train_label_pct = train_df["label"].value_counts(normalize=True).mul(100).rename("Train %")
    test_label_pct = test_df["label"].value_counts(normalize=True).mul(100).rename("Test %")
    label_comparison = pd.concat([train_label_pct, test_label_pct], axis=1).fillna(0).round(2)
    display(label_comparison)

if "attack_cat" in train_df.columns and "attack_cat" in test_df.columns:
    train_attack_pct = train_df["attack_cat"].dropna().value_counts(normalize=True).mul(100).rename("Train %")
    test_attack_pct = test_df["attack_cat"].dropna().value_counts(normalize=True).mul(100).rename("Test %")
    attack_comparison = pd.concat([train_attack_pct, test_attack_pct], axis=1).fillna(0).round(2)
    display(attack_comparison)

    attack_comparison.plot(kind="bar", figsize=(12, 6))
    plt.title("UNSW-NB15 — Training vs Testing Attack Distribution")
    plt.xlabel("Attack Category")
    plt.ylabel("Percentage of Attack Records")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_current_figure("04_train_vs_test_attack_distribution.png")
    plt.show()


## 25. Numerical feature distributions

Histograms are shown for a representative set of numerical features. The selection uses the first available traffic features and can be changed if a particular feature family needs more detailed inspection.

In [ ]:
selected_numeric = numeric_feature_columns[:10]
print("Selected numerical features:", selected_numeric)

for col in selected_numeric:
    plt.figure(figsize=(8, 4))
    plt.hist(train_df[col].dropna(), bins=50, edgecolor="black")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    save_current_figure(f"distribution_{re.sub(r'[^A-Za-z0-9._-]+', '_', col)}.png")
    plt.show()

## 26. Boxplots for selected numerical features

Boxplots provide a compact view of spread and potential extreme values. Potential outliers are **not removed** in this EDA.

In [ ]:
for col in selected_numeric:
    plt.figure(figsize=(8, 3))
    plt.boxplot(train_df[col].dropna(), vert=False)
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    save_current_figure(f"boxplot_{re.sub(r'[^A-Za-z0-9._-]+', '_', col)}.png")
    plt.show()

## 27. Outlier analysis using the IQR rule

The IQR rule is used only as a descriptive diagnostic. In network traffic, extreme observations can represent genuine attacks or unusual traffic behavior, so an IQR flag does not automatically mean the record is erroneous.

In [ ]:
outlier_rows = []

for col in numeric_feature_columns:
    q1 = train_df[col].quantile(0.25)
    q3 = train_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((train_df[col] < lower) | (train_df[col] > upper)).sum()

    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Outlier_Count": int(outliers),
        "Outlier_Percentage": outliers / len(train_df) * 100,
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values(
    "Outlier_Percentage", ascending=False
).reset_index(drop=True)

display(outlier_summary)

## 28. Correlation analysis

In [ ]:
correlation_matrix = train_df[numeric_feature_columns].corr()

plt.figure(figsize=(16, 12))
plt.imshow(correlation_matrix, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.index)), correlation_matrix.index)
plt.title("UNSW-NB15 Numerical Feature Correlation Matrix")
plt.tight_layout()
save_current_figure("correlation_matrix.png")
plt.show()

In [ ]:
if "label" in train_df.columns:
    correlations_with_label = (
        train_df[numeric_feature_columns + ["label"]]
        .corr()["label"]
        .drop("label")
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )
    display(correlations_with_label.to_frame("Correlation_with_Label"))
else:
    correlations_with_label = pd.Series(dtype=float)

## 29. Highly correlated feature pairs

Strong feature-feature correlations can indicate redundancy. They are reported here for later consideration; no feature is removed during EDA.

In [ ]:
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
strong_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "Feature_1", "level_1": "Feature_2", 0: "Correlation"})
)
strong_pairs["Absolute_Correlation"] = strong_pairs["Correlation"].abs()
strong_pairs = strong_pairs.sort_values("Absolute_Correlation", ascending=False)

print("Top correlated feature pairs:")
display(strong_pairs.head(20))

print("\nPairs with |correlation| >= 0.90:")
display(strong_pairs[strong_pairs["Absolute_Correlation"] >= 0.90])

## 30. Feature behaviour by attack category

To connect feature distributions with the project's attack-level analysis, a small set of numerical features is compared across attack categories. The selection uses the strongest absolute numerical association with the binary label when available.

In [ ]:
if "label" in train_df.columns and len(correlations_with_label):
    category_plot_features = correlations_with_label.abs().head(6).index.tolist()
else:
    category_plot_features = numeric_feature_columns[:6]

print("Features selected:", category_plot_features)

if "attack_cat" in train_df.columns:
    plot_df = train_df.copy()
    plot_df["attack_group"] = plot_df["attack_cat"].fillna("Normal").astype(str).str.strip()
    plot_df["attack_group"] = plot_df["attack_group"].replace("", "Normal")

    # Limit to the most frequent categories so the figures remain readable.
    top_categories = plot_df["attack_group"].value_counts().head(8).index
    plot_df = plot_df[plot_df["attack_group"].isin(top_categories)]

    for feature in category_plot_features:
        groups = []
        labels = []
        for category in top_categories:
            values = plot_df.loc[plot_df["attack_group"] == category, feature].dropna().values
            if len(values):
                groups.append(values)
                labels.append(category)

        if groups:
            plt.figure(figsize=(12, 6))
            plt.boxplot(groups, labels=labels)
            plt.title(f"{feature} by Attack Category")
            plt.xlabel("Attack Category")
            plt.ylabel(feature)
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            save_current_figure(f"feature_by_attack_{re.sub(r'[^A-Za-z0-9._-]+', '_', feature)}.png")
            plt.show()

## 31. Cardinality and constant-feature checks

In [ ]:
cardinality = pd.DataFrame({
    "Feature": train_df.columns,
    "Unique_Values": [train_df[c].nunique(dropna=False) for c in train_df.columns],
}).sort_values("Unique_Values", ascending=False).reset_index(drop=True)

display(cardinality)

constant_features = [c for c in train_df.columns if train_df[c].nunique(dropna=False) <= 1]
print("Constant features:", constant_features if constant_features else "None")

## 32. Train/test numerical distribution comparison

Mean, standard deviation, minimum, and maximum are compared as a first diagnostic. Large differences do not automatically imply a feature should be removed; they identify features that deserve further investigation.

In [ ]:
train_stats = train_df[numeric_feature_columns].describe().T
test_stats = test_df[numeric_feature_columns].describe().T

comparison_stats = pd.DataFrame({
    "Train_Mean": train_stats["mean"],
    "Test_Mean": test_stats["mean"],
    "Train_Std": train_stats["std"],
    "Test_Std": test_stats["std"],
    "Train_Min": train_stats["min"],
    "Test_Min": test_stats["min"],
    "Train_Max": train_stats["max"],
    "Test_Max": test_stats["max"],
})

comparison_stats["Mean_Difference_%"] = (
    (comparison_stats["Train_Mean"] - comparison_stats["Test_Mean"]).abs()
    / (comparison_stats["Train_Mean"].abs() + 1e-10)
) * 100

display(comparison_stats.sort_values("Mean_Difference_%", ascending=False))

## 33. Kolmogorov–Smirnov (KS) train/test distribution check

For each numerical feature, the two-sample KS statistic provides a quantitative measure of how different the empirical training and testing distributions are. A larger statistic indicates greater distributional separation.

This is an **EDA diagnostic**, not a reason by itself to alter the dataset.

In [ ]:
if SCIPY_AVAILABLE:
    ks_rows = []
    for col in numeric_feature_columns:
        train_values = train_df[col].dropna()
        test_values = test_df[col].dropna()
        statistic, p_value = ks_2samp(train_values, test_values)
        ks_rows.append({
            "Feature": col,
            "KS_Statistic": statistic,
            "P_Value": p_value,
        })

    ks_results = pd.DataFrame(ks_rows).sort_values(
        "KS_Statistic", ascending=False
    ).reset_index(drop=True)
    display(ks_results)
else:
    ks_results = pd.DataFrame()
    print("SciPy is not available; KS analysis was skipped.")

## 34. Exact train/test record-overlap check

Exact overlap is checked after excluding `id`. This is a diagnostic for potential train/test contamination. It does not detect near-duplicates.

In [ ]:
overlap_columns = [c for c in train_df.columns if c.lower() != "id"]

train_signatures = train_df[overlap_columns].astype(str).agg("|".join, axis=1)
test_signatures = test_df[overlap_columns].astype(str).agg("|".join, axis=1)

train_signature_set = set(train_signatures)
test_signature_set = set(test_signatures)
exact_overlap_count = len(train_signature_set.intersection(test_signature_set))

print(f"Unique exact train/test record overlap (excluding id): {exact_overlap_count:,}")
print("This is a diagnostic only; no rows are removed in this notebook.")

## 35. EDA findings relevant to NetShield

The project investigates why IDS models can perform well within one benchmark dataset but degrade across another. Therefore, the following EDA outputs are particularly important for the later cross-dataset experiments:

- **Feature representation:** what each UNSW-NB15 feature measures and which features have comparable semantics in CICIDS2017.
- **Traffic distribution:** ranges, skewness, zero prevalence, and train/test distribution differences.
- **Attack distribution:** normal/attack proportions and the frequency of individual attack families.
- **Class imbalance:** majority/minority ratios and minority attack categories.
- **Potential redundancy:** strongly correlated feature pairs.
- **Potential leakage risk:** exact train/test overlap diagnostics.

No modeling decision is made from a single EDA statistic. The next stage should use these observations to design a reproducible preprocessing and feature-engineering pipeline.

## 36. Save reusable EDA outputs

The following reusable EDA tables are saved to `NetShield/outputs/UNSW_NB15_EDA/` so later notebooks can reuse them without repeating the exploratory calculations. Important figures are saved in the `figures/` subfolder.

In [ ]:
# File inventory
csv_inventory = pd.DataFrame({
    "file": [str(p.name) for p in sorted(DATA_DIR.glob("*.csv"))],
    "size_MB": [p.stat().st_size / (1024**2) for p in sorted(DATA_DIR.glob("*.csv"))]
})
csv_inventory.to_csv(RESULTS_DIR / "file_inventory.csv", index=False)

# Compact dataset profile
dataset_profile = pd.DataFrame([
    ["training_rows", len(train_df)],
    ["testing_rows", len(test_df)],
    ["training_columns", train_df.shape[1]],
    ["testing_columns", test_df.shape[1]],
    ["numeric_features", len(numeric_feature_columns)],
    ["categorical_features", len(categorical_feature_columns)],
    ["unique_attack_categories", train_df["attack_cat"].nunique(dropna=True) if "attack_cat" in train_df.columns else np.nan],
    ["exact_duplicate_train", duplicate_full_train],
    ["exact_duplicate_test", duplicate_full_test],
    ["exact_train_test_overlap_ignoring_id", exact_overlap_count]
], columns=["metric", "value"])
dataset_profile.to_csv(RESULTS_DIR / "dataset_profile.csv", index=False)

feature_info.to_csv(RESULTS_DIR / "UNSW_feature_information.csv", index=False)
zero_summary.to_csv(RESULTS_DIR / "UNSW_zero_value_summary.csv", index=False)
skewness_table.to_csv(RESULTS_DIR / "UNSW_skewness_summary.csv")
outlier_summary.to_csv(RESULTS_DIR / "UNSW_outlier_summary.csv", index=False)
comparison_stats.to_csv(RESULTS_DIR / "UNSW_train_test_stats.csv")
correlation_matrix.to_csv(RESULTS_DIR / "UNSW_correlation_matrix.csv")
cardinality.to_csv(RESULTS_DIR / "UNSW_feature_cardinality.csv", index=False)

if not suspicious_summary.empty:
    suspicious_summary.to_csv(RESULTS_DIR / "UNSW_suspicious_categorical_values.csv", index=False)

if not attack_group_summary.empty:
    attack_group_summary.to_csv(RESULTS_DIR / "UNSW_normal_and_attack_distribution.csv")

if not attack_summary.empty:
    attack_summary.to_csv(RESULTS_DIR / "UNSW_attack_distribution.csv")

if 'label_comparison' in globals():
    label_comparison.to_csv(RESULTS_DIR / "UNSW_train_test_label_distribution.csv")

if 'attack_comparison' in globals():
    attack_comparison.to_csv(RESULTS_DIR / "UNSW_train_test_attack_distribution.csv")

if not ks_results.empty:
    ks_results.to_csv(RESULTS_DIR / "UNSW_train_test_KS_results.csv", index=False)

if feature_definitions is not None:
    feature_definitions.to_csv(RESULTS_DIR / "UNSW_feature_definitions_reference.csv", index=False)

print("UNSW-NB15 EDA tables saved to:", RESULTS_DIR.resolve())
print("UNSW-NB15 EDA figures saved to:", FIGURES_DIR.resolve())

# 37. Final automatic summary

The summary below is generated from the actual data when the notebook is run, so it does not rely on hard-coded dataset counts.

In [ ]:
print("=" * 90)
print("UNSW-NB15 EXPLORATION SUMMARY")
print("=" * 90)

print(f"\nTraining records: {len(train_df):,}")
print(f"Testing records:  {len(test_df):,}")
print(f"Columns in each split: {train_df.shape[1]:,}")
print(f"Numerical traffic features: {len(numeric_feature_columns):,}")
print(f"Categorical traffic features: {len(categorical_feature_columns):,}")

print(f"\nMissing training values: {int(train_df.isnull().sum().sum()):,}")
print(f"Missing testing values:  {int(test_df.isnull().sum().sum()):,}")
print(f"Infinite training values: {int(np.isinf(train_df[numeric_columns]).sum().sum()):,}")
print(f"Infinite testing values:  {int(np.isinf(test_df[numeric_columns]).sum().sum()):,}")
print(f"Exact duplicate rows — train: {duplicate_full_train:,}")
print(f"Exact duplicate rows — test:  {duplicate_full_test:,}")
print(f"Duplicate rows ignoring id — train: {duplicate_no_id_train:,}")
print(f"Duplicate rows ignoring id — test:  {duplicate_no_id_test:,}")
print(f"Exact train/test overlap excluding id: {exact_overlap_count:,}")

if "label" in train_df.columns:
    print("\nBinary label distribution:")
    print(train_df["label"].value_counts().sort_index().to_string())

if not attack_summary.empty:
    print("\nAttack-only category distribution:")
    print(attack_summary.to_string())

print("\nEDA completed successfully.")
print("=" * 90)

## 37. Output structure

After running the notebook, the UNSW-NB15 EDA results will be organized consistently with the CICIDS2017 EDA:

```text
NetShield/
└── outputs/
    ├── UNSW_NB15_EDA/
    │   ├── file_inventory.csv
    │   ├── dataset_profile.csv
    │   ├── UNSW_feature_information.csv
    │   ├── UNSW_feature_definitions_reference.csv
    │   ├── UNSW_zero_value_summary.csv
    │   ├── UNSW_skewness_summary.csv
    │   ├── UNSW_outlier_summary.csv
    │   ├── UNSW_train_test_stats.csv
    │   ├── UNSW_correlation_matrix.csv
    │   ├── UNSW_feature_cardinality.csv
    │   ├── UNSW_normal_and_attack_distribution.csv
    │   ├── UNSW_attack_distribution.csv
    │   ├── UNSW_train_test_label_distribution.csv
    │   ├── UNSW_train_test_attack_distribution.csv
    │   ├── UNSW_train_test_KS_results.csv
    │   └── figures/
    │       ├── 01_most_skewed_numerical_features.png
    │       ├── 02_normal_vs_attack_distribution.png
    │       ├── 03_normal_and_attack_categories.png
    │       ├── 04_train_vs_test_attack_distribution.png
    │       ├── correlation_matrix.png
    │       ├── distribution_*.png
    │       ├── boxplot_*.png
    │       └── feature_by_attack_*.png
    │
    └── CICIDS2017_EDA/
        └── ...
```

This means both datasets now have a dedicated EDA output directory and a dedicated figures directory.


# 38. Final interpretation

After running all cells, record the **observed** findings in your report using the generated tables and figures. The key conclusions should address:

1. **Dataset structure:** number of records, columns, numerical/categorical feature counts, and the availability of the predefined split.
2. **Data quality:** missing values, infinite values, duplicate records, suspicious placeholders, and constant features.
3. **Class imbalance:** normal-vs-attack ratio and the most/least represented attack families.
4. **Feature behaviour:** the most skewed features, high-zero features, large ranges, and prominent IQR outliers.
5. **Feature relationships:** highly correlated feature pairs and features most strongly associated with the binary label.
6. **Train/test differences:** features with the largest distribution differences and whether the target distribution is reasonably consistent.
7. **Leakage diagnostic:** whether exact train/test records overlap when `id` is ignored.
8. **Research implications:** which observations should inform preprocessing, feature alignment, class-imbalance handling, and later cross-dataset experiments.

> **Do not invent numerical findings before running the notebook.** The values in the final report should come directly from the outputs generated by this notebook.